### Mohammed Omar Mkhallati
### Data Science and its Applications
### Monday, September 21, 2026
### Practical Exercise: Data Scraping
### Source: https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html

#### Import Statements

In [1]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import ssl
import pandas as pd

#### Reading Table 2 From the FluWatch Page

In [2]:
ssl._create_default_https_context = ssl._create_unverified_context

url = 'https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html'

html = urlopen(url)
bs = BeautifulSoup(html, 'html.parser')

columns = [
    'Reporting Laboratory', 'Flu Tested', 'A(H1N1)pdm09 Positive', 'A(H3) Positive',
    'A(UnS) Positive', 'Total Flu A Positive', 'Total Flu B Positive', 'RSV Tested',
    'RSV Positive', 'PIV Tested', 'PIV 1 Positive', 'PIV 2 Positive', 'PIV 3 Positive',
    'PIV 4 Positive', 'Other PIV Positive', 'Adeno Tested', 'Adeno Positive',
    'hMPV Tested', 'hMPV Positive', 'Entero/Rhino Tested', 'Entero/Rhino Positive',
    'Coron Tested', 'Coron Positive'
]

# The page has two tables that start with a "Newfoundland" row: Table 1
# (one week) and Table 2 (the cumulative period). Matching on caption text
# turned out to be unreliable, so instead we collect every table that
# starts with Newfoundland and keep the LAST one, since Table 2 always
# comes after Table 1 in the page.
candidate_tables = []
for t in bs.find_all('table'):
    first_cell = t.find('td')
    if first_cell and first_cell.get_text(strip=True) == 'Newfoundland':
        candidate_tables.append(t)

table = candidate_tables[-1]

rows = []
for tr in table.find_all('tr'):
    cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
    if len(cells) == len(columns):
        rows.append(cells)

df = pd.DataFrame(rows, columns=columns)

#### Confirm the Data Was Read Properly

In [3]:
df.head()

,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
1,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
2,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
3,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
4,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6


#### Keeping One Row per Province/Territory

In [4]:
# Ontario, Quebec and Saskatchewan are reported as several individual labs
# that are summed into one bold row, so that row is used as the province total
province_map = {
    'Newfoundland': 'Newfoundland and Labrador',
    'Prince Edward Island': 'Prince Edward Island',
    'Nova Scotia': 'Nova Scotia',
    'New Brunswick': 'New Brunswick',
    'Province of Québec': 'Quebec',
    'Province of Ontario': 'Ontario',
    'Manitoba': 'Manitoba',
    'Province of Saskatchewan': 'Saskatchewan',
    'Province of Alberta': 'Alberta',
    'British Columbia': 'British Columbia',
    'Yukon': 'Yukon',
    'Northwest Territories': 'Northwest Territories',
    'Nunavut': 'Nunavut',
}

df = df[df['Reporting Laboratory'].isin(province_map)].copy()
df['Reporting Laboratory'] = df['Reporting Laboratory'].map(province_map)
df = df.rename(columns={'Reporting Laboratory': 'Province/Territory'})

order = list(province_map.values())
df['_sort'] = df['Province/Territory'].apply(order.index)
df = df.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)

#### Final Result

In [5]:
df

,Province/Territory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland and Labrador,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
4,Quebec,38395,0,0,5974,5975,81,35755,2816,11201,...,36,0,11288,373,10786,63,N.A.,N.A.,10703,310
5,Ontario,23795,752,180,546,1478,42,23518,1123,13600,...,38,0,13348,117,13307,101,3652,683,3080,117
6,Manitoba,4911,192,3,486,681,3,4828,101,2254,...,10,0,2254,25,1214,9,2254,310,1248,12
7,Alberta,18824,3132,73,1540,4745,29,15040,0,15040,...,154,0,15040,184,15040,189,15040,2693,15040,321
8,Yukon,143,16,1,6,23,0,122,2,N.A.,...,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.,N.A.
9,Nunavut,33,8,0,0,8,0,33,0,33,...,0,0,33,5,32,0,33,10,32,0
